# Full-page LightOnOCR — `pdf_files_3`

`extract_fulltext_lightonocr_pdf_files.ipynb`

Independent from the table pipeline (`*_lightonocr.json`).

Table pipeline (GLiNER/Ollama): PDF to `<stem>_lightonocr.json` in `data/pdf_files_3/` for `tables_only`.

This notebook: PDF to `<stem>_lightonocr_fulltext.json` in the same folder (optional; not used by current `text_only`, which uses GROBID TEI in `data/xml_files_3`).

Writes one JSON per PDF only. Does not overwrite table `*_lightonocr.json` files.


In [ ]:
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from pathlib import Path

AUTO_INSTALL = True
if AUTO_INSTALL:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "transformers", "pypdfium2", "Pillow",
    ])

import pypdfium2 as pdfium
import torch
from PIL import Image
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, here.parent]:
        if (p / "data" / "pwc_final.json").is_file():
            return p
    raise FileNotFoundError("Repo root not found (expected data/pwc_final.json)")


REPO = find_repo_root()
PDF_FILES_DIR = REPO / "data" / "pdf_files_3"
OUTPUT_SUFFIX = "_lightonocr_fulltext.json"  # distinto de *_lightonocr.json (tablas)

SHOW_VERBOSE = False
MAX_PAPERS = None  # prueba rápida: MAX_PAPERS = 3  (usa =, no :)
SKIP_EXISTING = True  # si el JSON ya existe, no re-OCR

OCR_MODEL_ID = "lightonai/LightOnOCR-2-1B"
OCR_TARGET_LONGEST = 1540
OCR_MAX_NEW_TOKENS = 8192

pdfs = sorted(PDF_FILES_DIR.glob("*.pdf"))
if MAX_PAPERS:
    pdfs = pdfs[:MAX_PAPERS]

print(f"Repo: {REPO}")
print(f"PDFs: {len(pdfs)} en {PDF_FILES_DIR}")
print(f"Salida: <stem>{OUTPUT_SUFFIX} (junto a cada PDF)")


In [ ]:
if torch.cuda.is_available():
    ocr_device = "cuda"
    ocr_dtype = torch.bfloat16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    ocr_device = "mps"
    ocr_dtype = torch.float32
else:
    ocr_device = "cpu"
    ocr_dtype = torch.float32

print(f"OCR device: {ocr_device}, dtype: {ocr_dtype}")

ocr_processor = LightOnOcrProcessor.from_pretrained(OCR_MODEL_ID)
ocr_model = LightOnOcrForConditionalGeneration.from_pretrained(
    OCR_MODEL_ID,
    torch_dtype=ocr_dtype,
    attn_implementation="eager",
).to(ocr_device)
print(f"Model loaded: {OCR_MODEL_ID}")


In [ ]:
def _pdf_readable(path_pdf: Path) -> tuple[bool, str]:
    if not path_pdf.is_file():
        return False, "file missing"
    try:
        if path_pdf.stat().st_size < 128:
            return False, "file too small"
        with open(path_pdf, "rb") as f:
            if f.read(5) != b"%PDF-":
                return False, "missing %PDF- header"
    except OSError as e:
        return False, str(e)
    return True, ""


def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = OCR_TARGET_LONGEST) -> Image.Image:
    page = pdf_doc[page_idx]
    bitmap = page.render(scale=200 / 72)
    pil_image = bitmap.to_pil()
    w, h = pil_image.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        pil_image = pil_image.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")
    return pil_image


def ocr_page(pil_image: Image.Image, max_new_tokens: int = OCR_MAX_NEW_TOKENS) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    pil_image.save(tmp, format="PNG")
    tmp.close()
    try:
        conversation = [{"role": "user", "content": [{"type": "image", "url": tmp.name}]}]
        inputs = ocr_processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            k: v.to(device=ocr_device, dtype=ocr_dtype) if v.is_floating_point() else v.to(ocr_device)
            for k, v in inputs.items()
        }
        with torch.no_grad():
            output_ids = ocr_model.generate(**inputs, max_new_tokens=max_new_tokens)
        generated_ids = output_ids[0, inputs["input_ids"].shape[1] :]
        return ocr_processor.decode(generated_ids, skip_special_tokens=True)
    finally:
        os.unlink(tmp.name)


def _count_html_tables(text: str) -> int:
    return len(re.findall(r"<table\b[^>]*>.*?</table>", text, flags=re.DOTALL | re.IGNORECASE))


def output_json_path(path_pdf: Path) -> Path:
    return path_pdf.parent / f"{path_pdf.stem}{OUTPUT_SUFFIX}"


def run_lightonocr_fulltext(path_pdf: Path, *, force: bool = False) -> dict:
    """OCR completo por página. Escribe un JSON junto al PDF. No toca *_lightonocr.json."""
    path_pdf = Path(path_pdf)
    stem = path_pdf.stem
    out_file = output_json_path(path_pdf)

    if not force and SKIP_EXISTING and out_file.is_file():
        with open(out_file, encoding="utf-8") as f:
            return json.load(f)

    ok, why = _pdf_readable(path_pdf)
    if not ok:
        raise ValueError(f"unreadable PDF ({why})")

    pdf_doc = pdfium.PdfDocument(str(path_pdf))
    pages_out = []
    try:
        for page_idx in range(len(pdf_doc)):
            page_num = page_idx + 1
            pil_image = render_pdf_page(pdf_doc, page_idx)
            ocr_text = ocr_page(pil_image)
            pages_out.append({
                "page": page_num,
                "text": ocr_text,
                "char_count": len(ocr_text),
                "n_html_tables": _count_html_tables(ocr_text),
            })
    finally:
        pdf_doc.close()

    full_text = "\n\n".join(
        f"--- page {p['page']} ---\n{p['text']}" for p in pages_out
    )
    result = {
        "file_name": str(path_pdf),
        "results": pages_out,
        "full_text": full_text,
    }
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result


print("Helpers ready.")


In [ ]:
t0_all = time.perf_counter()
n_total = len(pdfs)
ok_n = skip_n = err_n = 0

for i, pdf in enumerate(pdfs, start=1):
    stem = pdf.stem
    out_file = output_json_path(pdf)
    source = "skip" if out_file.is_file() and SKIP_EXISTING else "run"
    print(f"[{i}/{n_total}] {stem} ({source})")
    try:
        payload = run_lightonocr_fulltext(pdf)
        n_pages = len(payload.get("results", []))
        if source == "skip":
            skip_n += 1
        else:
            ok_n += 1
        print(f"  → {out_file.name} ({n_pages} pages)")
    except Exception as e:
        err_n += 1
        print(f"  SKIP: {e}")

print(f"\nDone in {time.perf_counter() - t0_all:.1f}s — new={ok_n}, skipped={skip_n}, errors={err_n}")
